# Notebook 03 — Operação Dia a Dia: Event Replay & Ciclo de Vida de Cache/TTL (`DAY = 649 .. 657`)

**Objetivo:** Demonstrar o **Modo B (Operação Contínua / Event Replay)** da arquitetura poliglota BasketLens:
1. **Baseline no Dia 649 (`prepare_replay_baseline(cutoff_day=649)`)**: estado inicial dos 3 bancos na véspera da janela operacional.
2. **Avanço Dia a Dia (`DAY = 650 .. 653`)**: leitura seletiva *out-of-core* via `pl.scan_csv().filter(pl.col("DAY") == day)` e contagem regressiva de TTL da Campanha 22 no Redis.
3. **Checkout e Resgate de Cupom por John Smith (`household_key = 208`) no Dia 654**:
   - Compra da cesta `41531980403` contendo `PRODUCT_ID = 1017772` e resgate do cupom `51800000050` (Campanha 22).
   - **Cassandra**: *append* transacional em `transactions_by_household` e `transactions_by_household_product`.
   - **MongoDB**: atualização incremental `$inc` em `behavior_summary` e rotação `$push` com `$slice: -5` em `recent_baskets`.
   - **Redis**: invalidação imediata do Cache-Aside (`DEL profile:208`) e baixa do cupom resgatado.
4. **Expiração Automática da Campanha 22 após o Dia 656 (`END_DAY = 656`)**: comprovação de `redis.ttl("campanha:22") <= 0` no Dia 657.

In [1]:
import pprint
from basketlens.connections import get_redis_client
from basketlens.queries import build_360_view, get_active_campaigns_for_household, get_household_product_history
from basketlens.replay import prepare_replay_baseline, replay_day, replay_range, restore_full_historical_state

print("1) Preparando baseline operacional no Dia 649 (véspera do Dia 650)...")
baseline = prepare_replay_baseline(cutoff_day=649, seconds_per_day=5)
pprint.pprint(baseline["household_208_baseline"])

print("\n2) Aquecendo o cache do perfil de John Smith (#208) via build_360_view(208):")
v_initial = build_360_view(208, limit_transactions=5)
v_warm = build_360_view(208, limit_transactions=5)
print("Primeira chamada (Cold):", v_initial["cache"])
print("Segunda chamada (Warm):", v_warm["cache"])
print("Última cesta no Dia 649:", v_warm["recent_baskets"][-1]["basket_id"], "(Dia", v_warm["recent_baskets"][-1]["day"], ")")

1) Preparando baseline operacional no Dia 649 (véspera do Dia 650)...


{'last_day': 647,
 'recent_basket_ids': [41124518352,
                       41160105604,
                       41259600218,
                       41338015594,
                       41439808495],
 'total_baskets': 121,
 'total_spend': 3877.19}

2) Aquecendo o cache do perfil de John Smith (#208) via build_360_view(208):


Primeira chamada (Cold): {'key': 'profile:208', 'cache_hit': False, 'source': 'mongodb', 'ttl_remaining_seconds': 60}
Segunda chamada (Warm): {'key': 'profile:208', 'cache_hit': True, 'source': 'redis', 'ttl_remaining_seconds': 60}
Última cesta no Dia 649: 41439808495 (Dia 647 )


## 2. Replay Operacional dos Dias 650 a 653 (Contagem Regressiva de TTL e Compra no Dia 650)

In [2]:
logs_650_653 = replay_range(
    start_day=650,
    end_day=653,
    seconds_per_day=5,
    prime_watched_cache_each_day=True,
)

print(f"{'DIA':<5} | {'TX_DIA':<7} | {'HH_ATIVOS':<10} | {'TTL_CAMP_22':<12} | {'208_COMPROU?':<13} | {'CACHE_DEL?':<11} | {'LAST_DAY_208':<12}")
print("-" * 85)
for entry in logs_650_653:
    w = entry["watched_household"]
    print(
        f"{entry['day']:<5} | "
        f"{entry['events']['transaction_rows']:<7} | "
        f"{entry['events']['distinct_households']:<10} | "
        f"{entry['redis']['campaign_22_ttl']:<12} | "
        f"{str(w['purchased_today']):<13} | "
        f"{str(w['cache_del_executed']):<11} | "
        f"{w['mongo_last_day']:<12}"
    )

DIA   | TX_DIA  | HH_ATIVOS  | TTL_CAMP_22  | 208_COMPROU?  | CACHE_DEL?  | LAST_DAY_208
-------------------------------------------------------------------------------------
650   | 3386    | 287        | 28           | True          | True        | 650         
651   | 4355    | 360        | 22           | False         | False       | 650         
652   | 4531    | 376        | 17           | False         | False       | 650         
653   | 3775    | 331        | 13           | False         | False       | 650         


## 3. Dia 654 — Compra de John Smith (`#208`), Resgate do Cupom `51800000050`, `$slice: -5` e Invalidação `DEL profile:208`

In [3]:
# Garante que profile:208 está em cache no Redis imediatamente antes do fechamento do Dia 654
_ = build_360_view(208, limit_transactions=5)
r_client = get_redis_client()
print("Estado ANTES do replay do Dia 654:")
print(" - profile:208 existe no Redis?", bool(r_client.exists("profile:208")))
print(" - Cupom 51800000050 disponível em household:208:coupons?", "51800000050" in r_client.smembers("household:208:coupons"))

day_654_report = replay_day(day=654, seconds_per_day=5, watch_household=208)
print("\nTelemetria do Dia 654 para household_key = 208:")
pprint.pprint(day_654_report["watched_household"])

print("\nEstado APÓS o replay do Dia 654:")
print(" - profile:208 existe no Redis logo após o DEL?", bool(r_client.exists("profile:208")))
print(" - Cupom 51800000050 ainda não resgatado?", "51800000050" in r_client.smembers("household:208:coupons"))
print(" - Histórico de cupons resgatados (household:208:redeemed_coupons):", r_client.smembers("household:208:redeemed_coupons"))
r_client.close()

# Nova chamada à Visão 360º: deve dar Cache Miss (lendo o novo estado do Dia 654 no MongoDB) e repopular o Redis
v_after_654 = build_360_view(208, limit_transactions=5)
print("\nChamada build_360_view(208) após invalidação:")
print(" - Cache status:", v_after_654["cache"])
print(" - MongoDB behavior_summary.last_day:", v_after_654["behavior_summary"]["last_day"])
print(" - Quantidade de cestas em recent_baskets ($slice: -5):", len(v_after_654["recent_baskets"]))
print(" - IDs das 5 últimas cestas em recent_baskets:", [b["basket_id"] for b in v_after_654["recent_baskets"]])
print(" - Transação mais recente no Cassandra:", v_after_654["recent_transactions"][0])

prod_1017772 = get_household_product_history(208, 1017772)
print(" - Histórico do produto 1017772 no Cassandra (compra mais recente no topo):", prod_1017772["purchases"][0])

Estado ANTES do replay do Dia 654:
 - profile:208 existe no Redis? True
 - Cupom 51800000050 disponível em household:208:coupons? True



Telemetria do Dia 654 para household_key = 208:
{'basket_ids_today': [41531980403],
 'cache_del_executed': True,
 'cache_existed_before_del': True,
 'cache_exists_after_del': False,
 'household_key': 208,
 'items_bought_today': 18,
 'mongo_last_day': 654,
 'mongo_recent_basket_ids': [41259600218,
                             41338015594,
                             41439808495,
                             41466570461,
                             41531980403],
 'mongo_recent_baskets_count': 5,
 'mongo_total_baskets': 123,
 'mongo_total_spend': 3935.09,
 'purchased_today': True,
 'redeemed_coupons_today': ['51800000050'],
 'spend_today': 48.03}

Estado APÓS o replay do Dia 654:
 - profile:208 existe no Redis logo após o DEL? False
 - Cupom 51800000050 ainda não resgatado? False
 - Histórico de cupons resgatados (household:208:redeemed_coupons): {'654:22:51800000050'}

Chamada build_360_view(208) após invalidação:
 - Cache status: {'key': 'profile:208', 'cache_hit': False, 'source': '

## 4. Dias 655 a 657 — Fim da Vigência e Expiração Automática da Campanha 22 (`END_DAY = 656`)

In [4]:
logs_655_657 = replay_range(start_day=655, end_day=657, seconds_per_day=2)
for entry in logs_655_657:
    print(
        f"Dia {entry['day']}: "
        f"Campanhas ativas={entry['redis']['active_campaign_ids']}, "
        f"Expiradas hoje={entry['redis']['expired_campaigns']}, "
        f"TTL campanha:22={entry['redis']['campaign_22_ttl']}"
    )

active_on_657 = get_active_campaigns_for_household(208)
print("\nCampanhas ativas para John Smith (#208) no Dia 657:")
pprint.pprint(active_on_657)

print("\nRestaurando o estado histórico completo (cutoff_day=711, reference_day=650) para análises subsequentes...")
restored = restore_full_historical_state(cutoff_day=711, reference_day=650, seconds_per_day=5)
print("Estado restaurado:", {"cutoff_day": restored["cutoff_day"], "reference_day": restored["reference_day"], "active_campaigns": list(restored["redis"]["active_campaigns"].keys())})

Dia 655: Campanhas ativas=[15, 20, 21, 22, 23], Expiradas hoje=[], TTL campanha:22=-2
Dia 656: Campanhas ativas=[15, 20, 21, 22, 23], Expiradas hoje=[], TTL campanha:22=-2
Dia 657: Campanhas ativas=[15, 20, 23], Expiradas hoje=[], TTL campanha:22=-2

Campanhas ativas para John Smith (#208) no Dia 657:
{'active_campaign_count': 0,
 'active_campaigns': [],
 'expired_or_inactive_campaigns': [],
 'household_key': 208,
 'latency_ms': 6.017,
 'redeemed_coupons_history': []}

Restaurando o estado histórico completo (cutoff_day=711, reference_day=650) para análises subsequentes...


Estado restaurado: {'cutoff_day': 711, 'reference_day': 650, 'active_campaigns': [15, 20, 21, 22, 23]}
